# H10 — A convolutional network and the matched filter compared

**Part VIII · the laboratory of lecture 20: the demonstration in class, Part B at home**

**Data.**

- `labs/data/gw150914_H1.hdf5`: the strain of the LIGO Hanford detector (H1), 32 s around the event GW150914 (GPS 1126259447 to 1126259479), sampled at 4096 Hz in the file. The file comes from the Gravitational Wave Open Science Center (GWOSC), release R1. `load_strain` reads it. It is the only **real** data of this laboratory. The setup cell measures the noise spectrum $S_{n}(f)$ on it and A6 analyzes it as the real record.
- The noise segments are **simulated**: Gaussian noise with the one-sided PSD $S_{n}(f)$ measured on the Hanford record, in segments of 1 s at 1024 Hz. A5 tilts the spectrum.
- The chirps are **simulated**: 16 waveforms `IMRPhenomD` of equal mass binaries, with component masses from 15 to 45 solar masses. They are whitened with the same spectrum and normalized to unit norm.
- The training set of 6000 segments and the test set of 2000 segments are **simulated**. Half of each has a chirp with an SNR $\varrho$ between 6 and 16 at a random arrival time. The label is $y = 1$ for a signal and $y = 0$ for noise only.
- The glitches of A4 are **simulated**: sine-Gaussians added to the noise. They are a model of what the instrument does and not a recording of it.

Everything runs offline. The whole notebook takes about two minutes on a laptop CPU.

**Libraries.**

- `torch`: the convolutional network, its training with the Adam optimizer and the scores.
- `numpy`: arrays, random numbers, the FFT for the coloring, the whitening and the matched filter.
- `scipy.signal`: `welch` estimates the noise spectrum and `decimate` changes the sampling rate in A6.
- `matplotlib.pyplot`: all the plots.
- `pycbc`: the waveform `IMRPhenomD` of the templates.
- `lal`: the waveform library that `pycbc` calls. The setup cell switches off the redirection of its output.
- `os` and `sys`: find the folder of the course, so that the two course modules can be imported.
- `gwcourse_signal` and `gwcourse_palette`: the functions of the course that load and whiten the strain. The second gives the colors.
- `tomllib`: reads `labs/config.toml`, the configuration of the laboratories: the folder of the data, the events and their files.

**What we do.**

- Part A, the demonstration: a labeled dataset of whitened chirps and noise, the matched filter as the reference, a convolutional network, then both methods with glitches in the noise, with a change of the noise spectrum, on the real record and against the SNR of the signal.
- Part B, on your own: change the SNR of the training set, the false alarm probability, the glitch rate, the size of the training set, the bank and the noise of the training set. Retrain and compare the efficiency.

> **References**
>
> (Gabbard et al. 2018), (George and Huerta 2018), (Sch\"afer and others 2023), (Cuoco et al. 2021)

## Tasks

The laboratory compares a convolutional network with the matched filter of H06 on the same problem, at the same false alarm probability $F$. The comparison is made with the efficiency $\epsilon$, the detection probability $D$ of lecture 13 at that $F$.

**Part A. The demonstration.** All the code is written out and runs cell by cell. Under each result we say what it shows.

- **A1.** The dataset: the functions `colored_noise`, `whiten_batch`, `chirp_bank`, `sine_gaussian` and `make_set`. One segment with a chirp and one without are plotted.
- **A2.** The matched filter as the reference: the statistic $\Phi$ maximized over the bank and over the arrival time, with the functions `matched_filter_statistic`, `threshold_at` and `efficiency`. Its efficiency at $F = 1$ %.
- **A3.** The network: the class `Net` and the functions `train`, `score` and `detection_curve`. Its efficiency at $F = 1$ % and the curve $D$ against $F$.
- **A4.** A population of simulated glitches in the noise: both methods on a glitchy test set, then the network retrained on glitches. The function `plot_efficiencies` is new.
- **A5.** A change of the noise spectrum: both methods on segments whitened with an outdated spectrum, for three tilts. The function is `drifted_set`.
- **A6.** The false alarm probability measurable on the real Hanford record: both statistics on one second segments and the rank of the segment of GW150914. The function is `event_rank`.
- **A7.** A claim to test: where does the network catch up? The efficiency against the SNR. The function is `efficiency_in_bins`.

**Part B. Your turn, at home.** The functions of Part A are given. We use them with our own choices. We retrain and compare.

- **B1.** Change the SNR range of the training set and retrain.
- **B2.** Change the false alarm probability.
- **B3.** Change the glitch rate of the test set.
- **B4.** Change the size of the training set.
- **B5.** Reduce the number of templates of the bank.
- **B6.** Optional: retrain the network on noise with a tilted spectrum.

In [ ]:
# libraries
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from scipy.signal import welch, decimate

# LAL redirects its printed output inside Jupyter and every waveform call becomes slow: switch it off
import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal
lal.swig_redirect_standard_output_error(False)

# the course root holds gwcourse_palette.py: find it from wherever we are
here = os.path.abspath('.')
while not os.path.exists(os.path.join(here, 'gwcourse_palette.py')):
    if os.path.dirname(here) == here:
        raise FileNotFoundError('gwcourse_palette.py not found: open the notebook from the course folder')
    here = os.path.dirname(here)
sys.path.insert(0, here)
sys.path.insert(0, os.path.join(here, 'labs'))

# the configuration of the laboratories: where the data are and the events we use
import tomllib
with open(os.path.join(here, 'labs', 'config.toml'), 'rb') as config_file:
    config = tomllib.load(config_file)
DATA_DIR = os.path.join(here, 'labs', config['data']['dir'])
GPS_GW150914 = config['events']['GW150914']['gps']
H1_FILE = os.path.join(DATA_DIR, config['events']['GW150914']['H1'])

# the course colors and the course functions that read and whiten the strain
from gwcourse_palette import P, apply_style
from gwcourse_signal import load_strain, whiten
apply_style()

# the pycbc function that gives the waveform of a binary
from pycbc.waveform import get_td_waveform

# the segments: one second at a sampling frequency of 1024 Hz
FS = 1024
N_SAMPLES = 1024

# the random numbers of the data and of the network: the seeds are fixed, so that the run can be repeated
rng = np.random.default_rng(0)
torch.manual_seed(0)

# one thread and deterministic kernels: the scores are then the same on every run
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)

# the real Hanford record
strain, times, fs = load_strain(H1_FILE)

# the one-sided PSD S_n(f) of the noise: Welch estimate on 4 s segments, on the frequency grid of a segment of 1 s
f_measured, p_measured = welch(strain, fs=fs, nperseg=int(4 * fs))
freqs = np.fft.rfftfreq(N_SAMPLES, d=1 / FS)
PSD = np.interp(freqs, f_measured, p_measured)

# nothing is used below 20 Hz: the spectrum there is set to its value at 20 Hz
index_20 = np.argmax(freqs >= 20)
PSD[freqs < 20] = PSD[index_20]
print(f"spectrum taken from {strain.size / fs:.0f} s of real H1 data")

## A1. The dataset

We need segments of noise, segments with a signal and a bank of templates. Each function does one step.

- `colored_noise` draws Gaussian noise with the one-sided PSD $S_{n}(f)$ measured on the Hanford record.
- `whiten_batch` divides each frequency by $\sqrt{S_{n}(f)}$, so that the noise has unit variance in each sample.
- `chirp_bank` builds sixteen whitened chirps of equal mass binaries, with component masses from 15 to 45 solar masses. Each template has unit norm, so that its amplitude factor in the data **is** the SNR $\varrho$.
- `sine_gaussian` is the model of a glitch, used from A4.
- `make_set` builds a labeled set. The segments with an odd index have the label $y = 1$ and contain a whitened chirp at a random arrival time, with $\varrho$ between 6 and 16. The others have $y = 0$.
- `note_box` writes a text in a box on a plot. We use it to put the numbers on the figures.

The cell after the functions builds two segments by hand at $\varrho = 8$, with and without a chirp. It plots them. The brick curve is the chirp alone.

In [ ]:
def colored_noise(n_examples, psd=PSD):
    """Draw segments of Gaussian noise with a given one-sided PSD.

    Args:
        n_examples: the number of segments.
        psd: the one-sided PSD S_n(f) on the grid freqs (default PSD, the one measured on H1).

    Returns:
        noise: an array of shape (n_examples, N_SAMPLES), the noise in the time domain.
    """
    # a complex Gaussian number for each frequency of each segment
    shape = (n_examples, len(freqs))
    real = rng.standard_normal(shape)
    imag = rng.standard_normal(shape)
    z = (real + 1j * imag) / np.sqrt(2)
    # the amplitude that the PSD gives at each frequency
    amplitude = np.sqrt(psd * FS * N_SAMPLES / 4)
    # back to the time domain
    noise = np.fft.irfft(z * amplitude, n=N_SAMPLES, axis=1)
    return noise


def whiten_batch(x, psd=PSD):
    """Whiten segments: divide each frequency by the noise amplitude there.

    Args:
        x: an array of shape (n_examples, N_SAMPLES), the segments.
        psd: the one-sided PSD S_n(f) used for the whitening (default PSD).

    Returns:
        whitened: the whitened segments, each one with unit variance.
    """
    # divide the Fourier transform by the square root of the PSD
    spectrum = np.fft.rfft(x, axis=1) / np.sqrt(psd)
    filtered = np.fft.irfft(spectrum, n=N_SAMPLES, axis=1)
    # unit variance for each segment
    whitened = filtered / filtered.std(axis=1, keepdims=True)
    return whitened


def chirp_bank(masses, f_low=30.0, psd=PSD):
    """Build a bank of whitened chirps of unit norm.

    Args:
        masses: the component masses of the equal mass binaries (solar masses, detector frame).
        f_low: the lower edge of the waveform (Hz, default 30).
        psd: the one-sided PSD S_n(f) used for the whitening (default PSD).

    Returns:
        bank: an array of shape (len(masses), N_SAMPLES), one template for each mass.
            The amplitude of a template in the data is then its SNR.
    """
    templates = []
    for m in masses:
        # the waveform of two equal masses m at 1 Mpc, sampled at FS
        hp, _ = get_td_waveform(approximant='IMRPhenomD', mass1=m, mass2=m,
                                delta_t=1 / FS, f_lower=f_low)
        h = np.asarray(hp)
        # the last second of the waveform, with zeros in front if it is shorter
        if len(h) >= N_SAMPLES:
            h = h[-N_SAMPLES:]
        else:
            h = np.pad(h, (N_SAMPLES - len(h), 0))
        # whiten with the PSD and normalize to unit norm
        hw = np.fft.irfft(np.fft.rfft(h) / np.sqrt(psd), n=N_SAMPLES)
        templates.append(hw / np.linalg.norm(hw))
    bank = np.array(templates)
    return bank


# the bank: sixteen equal mass binaries from 15 to 45 solar masses
BANK = chirp_bank(np.linspace(15, 45, 16))


def sine_gaussian(centre, frequency, q):
    """Compute a sine-Gaussian, the model of a glitch.

    Args:
        centre: the time of the maximum (s).
        frequency: the frequency of the oscillation (Hz).
        q: the quality factor, the number of cycles inside the envelope.

    Returns:
        glitch: the sine-Gaussian on the time axis of a segment, with a maximum of 1 at the centre.
    """
    # the time axis of a segment
    t = np.arange(N_SAMPLES) / FS
    # a Gaussian envelope times a cosine
    width = 2 * np.pi * frequency / q
    envelope = np.exp(-(t - centre) ** 2 * width ** 2 / 2)
    carrier = np.cos(2 * np.pi * frequency * (t - centre))
    glitch = envelope * carrier
    return glitch


def make_set(n_examples, snr_lo=6.0, snr_hi=16.0, glitch_rate=0.0, bank=BANK):
    """Build a labeled set of whitened segments.

    Args:
        n_examples: the number of segments. The odd ones have a chirp, the even ones do not.
        snr_lo: the smallest SNR of a chirp (default 6).
        snr_hi: the largest SNR of a chirp (default 16).
        glitch_rate: the fraction of the noise-only segments that carry a glitch (default 0).
        bank: the templates from which a chirp is drawn (default BANK).

    Returns:
        x32: the segments, an array of shape (n_examples, N_SAMPLES) of 32 bit floats.
        y: the labels, 1 for a segment with a chirp and 0 for noise only.
        snr: the SNR of the chirp in each segment, 0 where there is none.
    """
    # whitened noise for every segment
    x = whiten_batch(colored_noise(n_examples))
    y = np.zeros(n_examples, dtype=np.float32)
    snr = np.zeros(n_examples)
    for i in range(n_examples):
        if i % 2 == 1:
            # a chirp from the bank, with a random SNR and a random arrival time
            k = rng.integers(len(bank))
            snr[i] = rng.uniform(snr_lo, snr_hi)
            x[i] += snr[i] * np.roll(bank[k], int(rng.integers(-200, 60)))
            y[i] = 1.0
        elif rng.random() < glitch_rate:
            # a glitch with a random time, frequency and quality factor
            centre = rng.uniform(0.2, 0.8)
            frequency = rng.uniform(50, 300)
            q = rng.uniform(3, 15)
            glitch = sine_gaussian(centre, frequency, q)
            # unit norm times an amplitude between 8 and 25: the amplitude is its SNR
            amplitude = rng.uniform(8, 25)
            x[i] += amplitude * glitch / np.linalg.norm(glitch)
    x32 = x.astype(np.float32)
    return x32, y, snr


def note_box(ax, text, x=0.97, y=0.95, ha='right'):
    """Write a short text in a box, at a place of the axes that has no curve.

    Args:
        ax: the axes to write on.
        text: the text, with line breaks if needed.
        x: the horizontal position, as a fraction of the axes (default 0.97).
        y: the vertical position of the top of the box, as a fraction of the axes (default 0.95).
        ha: the horizontal alignment of the box, 'right' or 'left' (default 'right').
    """
    # a white box with a grey edge, so that the text is readable on the plot
    box = dict(boxstyle='round', facecolor='white', edgecolor=P['GREY_MID'])
    ax.text(x, y, text, transform=ax.transAxes, ha=ha, va='top', bbox=box)

In [ ]:
# two segments at SNR 8, built by hand: the same noise level, with and without a chirp
noise_two = whiten_batch(colored_noise(2))
chirp_8 = 8.0 * np.roll(BANK[8], -100)
segment_chirp = noise_two[0] + chirp_8
segment_noise = noise_two[1]

# the chirp with its noise above, the noise alone below; the chirp alone is drawn in brick
t = np.arange(N_SAMPLES) / FS
fig, axes = plt.subplots(2, 1, figsize=(9, 4.8), sharex=True, sharey=True)
axes[0].plot(t, segment_chirp, color=P['BLUE'], lw=0.8)
axes[0].plot(t, chirp_8, color=P['BRICK'], lw=1.2)
axes[0].set_ylabel('with a chirp')
axes[1].plot(t, segment_noise, color=P['GREY_DARK'], lw=0.8)
axes[1].set_ylabel('noise only')
axes[1].set_xlabel('time [s]')
axes[0].set_ylim(-6, 4)

# the numbers of the figure in a box, in an empty place
peak_chirp = np.abs(chirp_8).max()
peak_with = np.abs(segment_chirp).max()
peak_without = np.abs(segment_noise).max()
note_box(axes[0], f"chirp alone: peak {peak_chirp:.1f}\nlargest |x|: {peak_with:.1f}", x=0.02, ha='left', y=0.2)
note_box(axes[1], f"largest |x|: {peak_without:.1f}", x=0.02, ha='left', y=0.2)
plt.show()

# the whitened noise has unit variance by construction
check = whiten_batch(colored_noise(50))
print(f"bank of {len(BANK)} templates, norm of each {np.linalg.norm(BANK, axis=1).mean():.2f}; "
      f"whitened noise has unit variance by construction: {check.std():.3f}")
print(f"chirp alone: peak {peak_chirp:.2f}; largest |x| with the chirp {peak_with:.2f}, without {peak_without:.2f}")
print(f"energy of the chirp {np.sum(chirp_8 ** 2):.0f}, energy of the noise {np.sum(segment_noise ** 2):.0f}")

**What the result shows.** The chirp alone has a peak of 2.5, at the merger. Its energy is 64, against 1024 for the 1024 samples of noise. The largest sample is 3.1 in the segment with the chirp and 3.3 in the noise alone. The two panels cannot be told apart by eye at $\varrho = 8$: the chirp is below the noise at every time. The methods of this laboratory are needed for this reason. The whitened noise has unit variance (1.000) and the templates have unit norm.

## A2. The matched filter as the reference

We correlate each segment with every template of the bank, at every arrival time. We keep the largest value. The templates have unit norm and the whitened noise has unit variance, so the correlation is the statistic $\Phi$ of lecture 13, in units of the noise.

- `matched_filter_statistic` computes the largest $|\Phi|$ over the bank and over the arrival time, with the FFT.
- `threshold_at` finds the threshold $\Phi_{*}$ that a fraction $F$ of the noise-only segments crosses. $F$ is the false alarm probability.
- `efficiency` is the fraction of the signals above that threshold: the efficiency $\epsilon$, the detection probability $D$ at that $F$.

The second cell makes the training set of 6000 segments and the test set of 2000. It computes $\Phi$ on the test set and plots its distribution for the noise-only segments and for the signals, with the threshold at $F = 1$ %. All the other results of this laboratory are compared with this efficiency.

In [ ]:
def matched_filter_statistic(x, bank=BANK):
    """Compute the matched filter statistic of each segment.

    Args:
        x: an array of shape (n_examples, N_SAMPLES), the whitened segments.
        bank: the whitened templates of unit norm (default BANK).

    Returns:
        phi_max: the largest |Phi| over the bank and over the arrival time, one value for each segment.
    """
    # the Fourier transform of the segments and of the templates
    spectrum = np.fft.rfft(x, axis=1)
    bank_spectrum = np.fft.rfft(bank, axis=1)
    phi_max = np.zeros(len(x))
    for j in range(len(bank)):
        # the correlation with template j at every arrival time
        correlation = np.fft.irfft(spectrum * np.conj(bank_spectrum[j]),
                                   n=N_SAMPLES, axis=1)
        # the largest |Phi| of this template, kept if it is larger than the previous ones
        phi_template = np.abs(correlation).max(axis=1)
        phi_max = np.maximum(phi_max, phi_template)
    return phi_max


def threshold_at(stat, y, false_alarm=0.01):
    """Find the threshold that the noise-only segments cross with a given probability.

    Args:
        stat: the statistic of each segment, one value each.
        y: the labels, 0 for noise only and 1 for a signal.
        false_alarm: the false alarm probability F (default 0.01).

    Returns:
        threshold: the value reached by the largest F * N noise-only segments.
    """
    # the statistic of the noise-only segments, from the largest to the smallest
    noise = np.sort(stat[y == 0])
    noise = noise[::-1]
    # the position of the threshold in this list
    position = max(int(false_alarm * len(noise)) - 1, 0)
    threshold = noise[position]
    return threshold


def efficiency(stat, y, false_alarm=0.01):
    """Measure the efficiency at a given false alarm probability.

    Args:
        stat: the statistic of each segment, one value each.
        y: the labels, 0 for noise only and 1 for a signal.
        false_alarm: the false alarm probability F (default 0.01).

    Returns:
        fraction: the fraction of the signals above the threshold, the efficiency.
    """
    # the threshold is set on the noise-only segments
    threshold = threshold_at(stat, y, false_alarm=false_alarm)
    # the signals that are above it
    above = stat[y == 1] > threshold
    fraction = float(above.mean())
    return fraction

In [ ]:
# the training set and the test set
x_train, y_train, _ = make_set(6000)
x_test, y_test, snr_test = make_set(2000)

# the statistic of the matched filter on the test set, its threshold and its efficiency
mf_test = matched_filter_statistic(x_test)
threshold_mf = threshold_at(mf_test, y_test)
eff_mf = efficiency(mf_test, y_test)
print(f"matched filter: threshold {threshold_mf:.2f}, efficiency {eff_mf:.3f} at a false alarm probability F = 1 %")
noise_stat = mf_test[y_test == 0]
signal_stat = mf_test[y_test == 1]
print(f"noise only: median {np.median(noise_stat):.2f}, largest {noise_stat.max():.2f}; "
      f"signals: median {np.median(signal_stat):.2f}, smallest {signal_stat.min():.2f}")

# the statistic of the noise-only segments and of the signals, with the threshold
bins = np.linspace(3, 18, 61)
fig, ax = plt.subplots(figsize=(9, 4.4))
ax.hist(mf_test[y_test == 0], bins=bins, color=P['GREY_MID'], label='noise only')
ax.hist(mf_test[y_test == 1], bins=bins, color=P['BLUE'], alpha=0.8, label='with a chirp')
ax.axvline(threshold_mf, color=P['BRICK'], lw=1.5, ls='--')
ax.set_xlabel(r'matched filter statistic $\Phi$')
ax.set_ylabel('number of segments')
ax.legend(frameon=False, loc='upper right')
note_box(ax, f"threshold $\\Phi_{{*}}$ (dashed): {threshold_mf:.2f}\nF = 1 %\nefficiency: {eff_mf:.3f}", x=0.97, y=0.7)
plt.show()

**What the result shows.** The noise-only segments have a median $\Phi$ of 3.42 and the largest is 5.79. The threshold for $F = 1$ % is $\Phi_{*} = 4.59$. It is the value of the tenth largest of the 1000 noise-only segments. The signals have a median of 11.21 and the smallest is 4.73, so all of them are above the threshold. The efficiency is 1.000.

| quantity | value |
|---|---|
| threshold $\Phi_{*}$ at $F = 1$ % | 4.59 |
| median $\Phi$, noise only | 3.42 |
| median $\Phi$, signals | 11.21 |
| efficiency $\epsilon$ | 1.000 |

The efficiency is the highest possible. The dataset satisfies every assumption of lecture 13: Gaussian noise, stationary, with a known spectrum and a signal that is in the bank. The filter is optimal here and no method can do better. The threshold is lower than the $\Phi_{*} = 8$ of H06 because $F = 1$ % is a much larger false alarm probability than a detection claim needs. A6 comes back to this.

## A3. The network

The network is a small one-dimensional convolutional network: three convolutions with pooling, a maximum over time and one linear output. It has 17009 weights.

- `Net` is the class of the network.
- `train` trains it with binary cross entropy and the Adam optimizer: ten passes over the 6000 segments of the training set.
- `score` gives the output of the network for each segment. It is the ranking statistic of the network, as $\Phi$ is for the filter.
- `detection_curve` measures $\epsilon$ at several false alarm probabilities, which gives the curve $D$ against $F$.

We score the test set of A2 and compare the two methods at $F = 1$ %. The plot shows $D$ against $F$ for both.

In [ ]:
class Net(nn.Module):
    """A small convolutional network: three convolutions with pooling and one linear output."""

    def __init__(self):
        """Build the layers: the convolutions of the body and the linear layer of the head."""
        super().__init__()
        # three convolutions, each with a ReLU, and a maximum over time at the end
        self.body = nn.Sequential(
            nn.Conv1d(1, 16, 32, stride=2), nn.ReLU(), nn.MaxPool1d(4),
            nn.Conv1d(16, 32, 16), nn.ReLU(), nn.MaxPool1d(4),
            nn.Conv1d(32, 32, 8), nn.ReLU(), nn.AdaptiveMaxPool1d(1))
        # one output: the score of the segment
        self.head = nn.Linear(32, 1)

    def forward(self, x):
        """Compute the score of each segment.

        Args:
            x: a tensor of shape (n_examples, 1, N_SAMPLES), the whitened segments.

        Returns:
            logits: a tensor of shape (n_examples,), the output before the logistic function.
        """
        # the 32 features that the convolutions find
        features = self.body(x)
        features = features.squeeze(-1)
        # one score for each segment
        logits = self.head(features)
        logits = logits.squeeze(-1)
        return logits


def train(x, y, epochs=10, seed=0, learning_rate=2e-3, batch_size=128):
    """Train the network with binary cross entropy and the Adam optimizer.

    Args:
        x: the segments, an array of shape (n_examples, N_SAMPLES).
        y: the labels, 1 for a signal and 0 for noise only.
        epochs: the number of passes over the training set (default 10).
        seed: the seed of the initial weights and of the order of the examples (default 0).
        learning_rate: the step of the optimizer (default 2e-3).
        batch_size: the number of examples of each step (default 128).

    Returns:
        net: the trained network.
    """
    # the seed fixes the initial weights and the order of the examples
    torch.manual_seed(seed)
    net = Net()
    optimizer = torch.optim.Adam(net.parameters(), lr=learning_rate)
    loss_fn = nn.BCEWithLogitsLoss()
    # the network wants a channel axis: shape (n_examples, 1, N_SAMPLES)
    inputs = torch.from_numpy(x).unsqueeze(1)
    labels = torch.from_numpy(y)
    for epoch in range(epochs):
        # a new random order of the examples at each pass
        order = torch.randperm(len(inputs))
        for start in range(0, len(inputs), batch_size):
            # one step on a batch: the loss, its gradient and the update of the weights
            batch = order[start:start + batch_size]
            optimizer.zero_grad()
            loss = loss_fn(net(inputs[batch]), labels[batch])
            loss.backward()
            optimizer.step()
    return net


def score(net, x):
    """Compute the score of each segment with a trained network.

    Args:
        net: the trained network.
        x: the segments, an array of shape (n_examples, N_SAMPLES).

    Returns:
        scores: an array with one score for each segment, the ranking statistic.
    """
    # no gradient is needed to score
    with torch.no_grad():
        inputs = torch.from_numpy(x).unsqueeze(1)
        output = net(inputs)
    scores = output.numpy()
    return scores


def detection_curve(stat, y, false_alarms):
    """Measure the efficiency at several false alarm probabilities: the curve D against F.

    Args:
        stat: the statistic of each segment, one value each.
        y: the labels, 0 for noise only and 1 for a signal.
        false_alarms: the false alarm probabilities F, a list of numbers.

    Returns:
        detection: the efficiency at each F, an array as long as false_alarms.
    """
    detection = []
    for false_alarm in false_alarms:
        detection.append(efficiency(stat, y, false_alarm=false_alarm))
    detection = np.array(detection)
    return detection

In [ ]:
# train on the 6000 segments of the training set and score the test set
net = train(x_train, y_train)
cnn_test = score(net, x_test)
n_weights = sum(p.numel() for p in net.parameters())

# the efficiency of the two methods at the same false alarm probability
eff_mf = efficiency(mf_test, y_test)
eff_net = efficiency(cnn_test, y_test)
print(f"network with {n_weights} weights")
print(f"matched filter {eff_mf:.3f}   network {eff_net:.3f}")
d_net = detection_curve(cnn_test, y_test, (0.01, 0.1, 1.0))
print(f"network: D at F = 1 %, 10 % and 100 %: {d_net[0]:.3f}, {d_net[1]:.3f}, {d_net[2]:.3f}")

# the detection probability D against the false alarm probability F, for both methods
false_alarms = np.logspace(-2, 0, 25)
curve_mf = detection_curve(mf_test, y_test, false_alarms)
curve_net = detection_curve(cnn_test, y_test, false_alarms)
fig, ax = plt.subplots(figsize=(9, 4.4))
ax.semilogx(false_alarms, curve_mf, color=P['BRICK'], lw=1.8, label='matched filter')
ax.semilogx(false_alarms, curve_net, color=P['BLUE'], lw=1.8, label='network')
ax.axvline(0.01, color=P['GREY_MID'], lw=1, ls='--')
ax.set_xlabel('false alarm probability $F$')
ax.set_ylabel('detection probability $D$')
ax.set_ylim(0.9, 1.01)
ax.legend(frameon=False, loc='lower right')
note_box(ax, f"at F = 1 % (dashed)\nmatched filter: {eff_mf:.3f}\nnetwork: {eff_net:.3f}", x=0.4, y=0.55, ha='left')
plt.show()

**What the result shows.** The network reaches 0.944 against 1.000 for the filter at $F = 1$ %. Its efficiency is lower, as expected: the filter is optimal here and the network has to learn from 6000 examples what the filter is given. The curve shows that the gap is largest at small $F$. The network has $D = 0.944$ at $F = 1$ %, 0.982 at 10 % and 1.000 at 100 %.

The last digit is uncertain. The threshold at 1 % rests on the loudest ten of 1000 noise-only segments, so every efficiency in this laboratory is uncertain by a few hundredths. The network runs on one thread with deterministic operations, so the numbers repeat from run to run. On another processor or with another version of `torch` the last digit can change.

**This result is a check on any experiment of this kind.** A network that does better than a matched filter on Gaussian noise with a known waveform indicates an error in the comparison.

## A4. A population of glitches in the noise

Real detector noise has transients. We rebuild the **test** set with a sine-Gaussian glitch in half of the noise-only segments. Its time is random, its frequency is between 50 and 300 Hz, its quality factor between 3 and 15 and its amplitude, which is its SNR, between 8 and 25. These are the blips of H06, simulated. We do not retrain: the network is the one of A3, which has not seen glitches. Then we retrain on a training set that also has glitches.

The filter runs here alone, in one detector, with no coincidence. The cell shows what glitches do to the filter itself. `plot_efficiencies` draws the efficiency of the two methods as pairs of bars. In the left plot there is one glitch of SNR 15 in whitened noise.

In [ ]:
def plot_efficiencies(labels, eff_mf, eff_net, ax=None):
    """Draw the efficiency of the two methods as pairs of bars, with the values above the bars.

    Args:
        labels: the name of each case, a list of strings.
        eff_mf: the efficiency of the matched filter in each case, a list of numbers.
        eff_net: the efficiency of the network in each case, a list of numbers.
        ax: the axes to draw on (default None: the current axes).
    """
    if ax is None:
        ax = plt.gca()
    # one pair of bars for each case
    positions = np.arange(len(labels))
    ax.bar(positions - 0.2, eff_mf, width=0.38, color=P['BRICK'], label='matched filter')
    ax.bar(positions + 0.2, eff_net, width=0.38, color=P['BLUE'], label='network')
    # the value of each bar, just above it
    for position, value in zip(positions, eff_mf):
        ax.text(position - 0.2, value + 0.02, f"{value:.2f}", ha='center')
    for position, value in zip(positions, eff_net):
        ax.text(position + 0.2, value + 0.02, f"{value:.2f}", ha='center')
    ax.set_xticks(positions)
    ax.set_xticklabels(labels)
    ax.set_ylabel(r'efficiency $\epsilon$ at $F = 1$ %')
    ax.set_ylim(0, 1.3)
    ax.legend(frameon=False, loc='upper center', ncol=2)

In [ ]:
# a test set with glitches in half of the noise-only segments
x_glitchy, y_glitchy, _ = make_set(2000, glitch_rate=0.5)

# the statistic of the matched filter on the glitchy set: the filter runs alone, with no other test
mf_glitchy = matched_filter_statistic(x_glitchy)

# the efficiency on clean noise, then with glitches, for the network of A3: it has not seen glitches
eff_clean_mf = efficiency(mf_test, y_test)
eff_clean_net = efficiency(cnn_test, y_test)
eff_glitch_mf = efficiency(mf_glitchy, y_glitchy)
eff_glitch_net = efficiency(score(net, x_glitchy), y_glitchy)

# a training set with glitches, a network trained on it and its efficiency on the glitchy set
x_train2, y_train2, _ = make_set(6000, glitch_rate=0.5)
net_glitch = train(x_train2, y_train2, seed=1)
eff_retrained = efficiency(score(net_glitch, x_glitchy), y_glitchy)

threshold_clean = threshold_at(mf_test, y_test)
threshold_glitch = threshold_at(mf_glitchy, y_glitchy)
print(f"threshold of the filter: {threshold_clean:.2f} on clean noise, {threshold_glitch:.2f} with glitches")
print(f"{'':34} {'MF':>6} {'network':>9}")
print(f"{'clean noise':34} {eff_clean_mf:6.3f} {eff_clean_net:9.3f}")
print(f"{'with glitches, network unchanged':34} {eff_glitch_mf:6.3f} {eff_glitch_net:9.3f}")
print(f"{'with glitches, network retrained':34} {eff_glitch_mf:6.3f} {eff_retrained:9.3f}")

# left: one glitch of SNR 15 in whitened noise; right: the efficiencies
t = np.arange(N_SAMPLES) / FS
glitch = sine_gaussian(0.5, 150.0, 8.0)
segment_glitch = whiten_batch(colored_noise(1))[0] + 15.0 * glitch / np.linalg.norm(glitch)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), gridspec_kw={'width_ratios': [1, 1.3]})
axes[0].plot(t, segment_glitch, color=P['GREY_DARK'], lw=0.8)
axes[0].set_xlabel('time [s]')
axes[0].set_ylabel('whitened noise with a glitch')
plot_efficiencies(['clean', 'glitches', 'glitches,\nretrained'],
                  [eff_clean_mf, eff_glitch_mf, eff_glitch_mf],
                  [eff_clean_net, eff_glitch_net, eff_retrained], ax=axes[1])
plt.tight_layout()
plt.show()

**What the result shows.** The filter falls from 1.000 to 0.726. The network does not fall: 0.944 on clean noise and 0.960 with the glitches. The difference of 0.016 is inside the uncertainty of the measure. The retrained network gives 0.960 too.

| | matched filter | network |
|---|---|---|
| clean noise | 1.000 | 0.944 |
| glitches, network unchanged | 0.726 | 0.960 |
| glitches, network retrained | 0.726 | 0.960 |

The filter falls because its background is now full of transients that correlate with some template. The threshold at $F = 1$ % moves from 4.59 to 9.04 and the weak signals fall below it. A real search rejects many glitches with the coincidence of two detectors, as in H06. Here the filter runs alone, in one detector, to show the effect of the glitches on the filter itself.

**The retraining changes nothing at this precision.** The network was not trained with glitches and it kept its efficiency. A blip does not look like a chirp under the features that the network learned. This robustness is not general: it holds for these glitches.

## A5. A change of the noise spectrum

We leave out the glitches and change the noise itself. The test segments have noise with the spectrum $S_{n}(f)$ times a power law $(f/50\ \mathrm{Hz})^{-\alpha}$, with the tilt $\alpha = 0$, 0.3 and 0.6. We whiten them with the **old** spectrum $S_{n}(f)$. This is the effect of an outdated whitening filter when the detector changes over weeks. The network is the one of A3, not retrained.

`drifted_set` builds the segments. In the left plot, the ratio of the spectrum of the noise to the spectrum used for the whitening, for each tilt. In the right plot, the efficiency of both methods.

In [ ]:
def drifted_set(n_examples, tilt, snr_lo=6.0, snr_hi=16.0, bank=BANK):
    """Build a labeled set with noise of a tilted spectrum, whitened with the old spectrum.

    Args:
        n_examples: the number of segments. The odd ones have a chirp, the even ones do not.
        tilt: the exponent of the power law that multiplies the PSD around 50 Hz (0 leaves it unchanged).
        snr_lo: the smallest SNR of a chirp (default 6).
        snr_hi: the largest SNR of a chirp (default 16).
        bank: the templates from which a chirp is drawn (default BANK).

    Returns:
        x32: the segments, an array of shape (n_examples, N_SAMPLES) of 32 bit floats.
        y: the labels, 1 for a segment with a chirp and 0 for noise only.
    """
    # the spectrum of the noise now: the old one times a power law
    psd_now = PSD * (np.maximum(freqs, 1.0) / 50.0) ** (-tilt)
    # noise with the spectrum now, whitened with the old spectrum
    noise = colored_noise(n_examples, psd=psd_now)
    x = whiten_batch(noise, psd=PSD)
    y = np.zeros(n_examples, dtype=np.float32)
    for i in range(n_examples):
        if i % 2 == 1:
            # a chirp from the bank, with a random SNR and a random arrival time
            k = rng.integers(len(bank))
            snr = rng.uniform(snr_lo, snr_hi)
            x[i] += snr * np.roll(bank[k], int(rng.integers(-200, 60)))
            y[i] = 1.0
    x32 = x.astype(np.float32)
    return x32, y

In [ ]:
# the efficiency of the two methods for each tilt
tilts = (0.0, 0.3, 0.6)
drift_mf = []
drift_net = []
print(f"{'tilt':>6} {'MF':>7} {'network':>9}")
for tilt in tilts:
    xd, yd = drifted_set(2000, tilt)
    drift_mf.append(efficiency(matched_filter_statistic(xd), yd))
    drift_net.append(efficiency(score(net, xd), yd))
    print(f"{tilt:>6.1f} {drift_mf[-1]:7.3f} {drift_net[-1]:9.3f}")

# how much each one loses from tilt 0 to tilt 0.6
loss_mf = drift_mf[0] - drift_mf[-1]
loss_net = drift_net[0] - drift_net[-1]
print(f"loss: filter {loss_mf:.3f}, network {loss_net:.3f}, "
      f"{loss_net / loss_mf:.1f} times more for the network")

# left: the spectrum of the noise now divided by the spectrum used to whiten; right: the efficiencies
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
colors = (P['GREY_DARK'], P['AMBER'], P['BRICK'])
for tilt, color in zip(tilts, colors):
    ratio = (np.maximum(freqs, 1.0) / 50.0) ** (-tilt)
    axes[0].plot(freqs[freqs >= 20], ratio[freqs >= 20], color=color, lw=1.8, label=f"tilt {tilt:.1f}")
axes[0].set_xlabel('frequency [Hz]')
axes[0].set_ylabel(r'$S_{n}$ now / $S_{n}$ of the whitening')
axes[0].legend(frameon=False, loc='upper right')
axes[1].plot(tilts, drift_mf, 'o-', color=P['BRICK'], lw=1.8, label='matched filter')
axes[1].plot(tilts, drift_net, 'o-', color=P['BLUE'], lw=1.8, label='network')
axes[1].set_xlabel('tilt')
axes[1].set_ylabel(r'efficiency $\epsilon$ at $F = 1$ %')
axes[1].set_ylim(0.2, 1.05)
axes[1].legend(frameon=False, loc='center left')
note_box(axes[1], f"loss, tilt 0 to 0.6\nfilter: {loss_mf:.3f}\nnetwork: {loss_net:.3f}", x=0.97, y=0.27)
plt.tight_layout()
plt.show()

**What the result shows.** The filter goes 0.999, 0.975, 0.948 for the tilts 0, 0.3 and 0.6. The network goes 0.904, 0.773, 0.493. From the tilt 0 to 0.6 the filter loses 0.051 and the network loses 0.411.

**The network loses 8.1 times more.** The filter states its assumptions explicitly: a wrong spectrum reduces its output only by the mismatch that it causes. The network has its assumptions in the training set. Its response to data that differ from the training set is not guaranteed.

A4 and A5 together: **where the noise contains something that cannot be written down, the network is better. Where the noise differs from the training noise, the filter is better.** A real search uses the classical pipeline and estimates its spectrum again continuously, which is the same statement in operational terms.

## A6. The false alarm probability on the real record

We cut the real Hanford record into one second segments and run both statistics on them. The record is whitened as in H07 with `whiten` and brought to 1024 Hz with `decimate`. Two seconds are cut at each end, where the whitening filter rings. Each segment has zero mean and unit variance. The segment that contains GW150914 is the one with the merger.

`event_rank` gives the place of a segment when all the segments are ordered by a statistic: 1 is the largest. We also compute how many one second segments a false alarm rate of one per century needs: the false alarm probability of one segment is then $F = 1/(\text{number of seconds in a century})$.

In [ ]:
def event_rank(stat, segment):
    """Find the place of one segment when all segments are ordered by a statistic.

    Args:
        stat: the statistic of each segment, one value each.
        segment: the index of the segment.

    Returns:
        rank: the place of the segment, 1 for the largest value of the statistic.
    """
    # the indices of the segments, from the largest value to the smallest
    order = np.argsort(-stat)
    # the position of the chosen segment in this list, counted from 1
    place = np.where(order == segment)[0][0]
    rank = int(place) + 1
    return rank

In [ ]:
# whiten the whole record the way H07 did, then bring it to the band of this laboratory
whitened = whiten(strain, fs)
w = decimate(whitened, int(fs // FS), zero_phase=True)

# cut two seconds at each end: the whitening filter rings there
edge = 2 * FS
w = w[edge:-edge]
start_gps = times[0] + 2.0

# cut the record into one second segments
n_seconds = len(w) // N_SAMPLES
segments = w[:n_seconds * N_SAMPLES].reshape(n_seconds, N_SAMPLES).astype(np.float32)

# zero mean and unit variance for each segment
mean = segments.mean(axis=1, keepdims=True)
std = segments.std(axis=1, keepdims=True)
segments = (segments - mean) / std

# the segment that contains the event
event_segment = int((GPS_GW150914 - start_gps) // 1)
print(f"{n_seconds} one second segments of real data, the event in number {event_segment}")
print(f"smallest false alarm probability they can measure: 1/{n_seconds} = {1 / n_seconds:.3f}")

# the number of one second segments that a false alarm rate of one per century needs
seconds_per_century = 100 * 365.25 * 24 * 3600
print(f"one false alarm per century: F = {1 / seconds_per_century:.1e}, "
      f"that is {seconds_per_century:.1e} segments")

# the statistic of each method on every segment and the rank of the event
stat_mf = matched_filter_statistic(segments)
stat_net = score(net, segments)
rank_mf = event_rank(stat_mf, event_segment)
rank_net = event_rank(stat_net, event_segment)
print(f"  {'matched filter':15s} ranks the event's segment {rank_mf} of {n_seconds}")
print(f"  {'network':15s} ranks the event's segment {rank_net} of {n_seconds}")
others = np.delete(stat_mf, event_segment)
print(f"statistic of the filter: event {stat_mf[event_segment]:.2f}, "
      f"largest of the other segments {others.max():.2f}, median {np.median(others):.2f}")

# the statistic of every segment, the event's segment in brick
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, stat, rank, name in ((axes[0], stat_mf, rank_mf, 'matched filter statistic'),
                             (axes[1], stat_net, rank_net, 'network score')):
    ax.plot(np.arange(n_seconds), stat, 'o', color=P['GREY_MID'])
    ax.plot(event_segment, stat[event_segment], 'o', color=P['BRICK'], markersize=9)
    # room above the points for the box
    low, high = ax.get_ylim()
    ax.set_ylim(low, high + 0.6 * (high - low))
    ax.set_xlabel('segment of one second')
    ax.set_ylabel(name)
    note_box(ax, f"event: segment {event_segment}\nrank {rank} of {n_seconds}", x=0.03, y=0.95, ha='left')

# the threshold of A2 on the plot of the filter
axes[0].axhline(threshold_mf, color=P['BRICK'], lw=1.2, ls='--')
note_box(axes[0], f"dashed: threshold of A2, {threshold_mf:.2f}", x=0.03, y=0.78, ha='left')
plt.tight_layout()
plt.show()

**What the result shows.** The record gives 28 segments. They cannot measure a false alarm probability below 1/28, that is 3.6 %. The 1 % used in this laboratory is already beyond what this record supports. A false alarm rate of one per century needs $F = 3.2\times10^{-10}$, that is $3.2\times10^{9}$ one second segments. A real search builds this background with time slides for this reason.

| | matched filter | network |
|---|---|---|
| rank of the segment of GW150914 | 1 of 28 | 8 of 28 |

The filter ranks the segment of GW150914 first, in a record on which it was not tuned. Its statistic is 5.54, above the threshold 4.59 of A2, the dashed line of the plot. The largest of the other segments is 4.54, below it. The network ranks the segment eighth. A likely reason is the effect of A5: the real record differs from the noise on which the network was trained. The filter assumes only the spectrum and finds the event.

## A7. A claim to test: where does the network catch up?

We plot the efficiency of both methods against the injected SNR $\varrho$, in bins, on the clean test set of A2. The threshold is the same for all the bins, at $F = 1$ %. `efficiency_in_bins` gives the center of each bin and the efficiency in it.

In [ ]:
def efficiency_in_bins(stat, y, snr, edges, false_alarm=0.01, min_count=20):
    """Measure the efficiency in bins of the injected SNR.

    Args:
        stat: the statistic of each segment, one value each.
        y: the labels, 0 for noise only and 1 for a signal.
        snr: the injected SNR of each segment.
        edges: the edges of the bins of the SNR.
        false_alarm: the false alarm probability F of the threshold (default 0.01).
        min_count: the smallest number of signals in a bin that is kept (default 20).

    Returns:
        centres: the center of each bin that is kept.
        eff: the efficiency in each of these bins.
    """
    # one threshold for all the bins, set on the noise-only segments
    threshold = threshold_at(stat, y, false_alarm=false_alarm)
    centres = []
    eff = []
    for low, high in zip(edges[:-1], edges[1:]):
        # the signals with an injected SNR inside this bin
        in_bin = (y == 1) & (snr >= low) & (snr < high)
        # a bin with too few signals is left out
        if in_bin.sum() >= min_count:
            centres.append(0.5 * (low + high))
            eff.append(float((stat[in_bin] > threshold).mean()))
    return centres, eff

In [ ]:
# the bins of the injected SNR, from 6 to 16 in steps of 2, and the efficiency of each method in them
edges = np.arange(6, 17, 2.0)
centres, bins_mf = efficiency_in_bins(mf_test, y_test, snr_test, edges)
centres, bins_net = efficiency_in_bins(cnn_test, y_test, snr_test, edges)
print(f"{'SNR bin centre':15s} " + "  ".join(f"{c:5.0f}" for c in centres))
print(f"{'matched filter':15s} " + "  ".join(f"{e:5.2f}" for e in bins_mf))
print(f"{'network':15s} " + "  ".join(f"{e:5.2f}" for e in bins_net))

# the efficiency against the injected SNR
fig, ax = plt.subplots(figsize=(9, 4.4))
ax.plot(centres, bins_mf, 'o-', color=P['BRICK'], lw=1.8, label='matched filter')
ax.plot(centres, bins_net, 'o-', color=P['BLUE'], lw=1.8, label='network')
ax.set_xlabel(r'injected SNR $\varrho$')
ax.set_ylabel(r'efficiency $\epsilon$ at $F = 1$ %')
ax.set_ylim(0.5, 1.05)
ax.legend(frameon=False, loc='lower right')
gap = bins_mf[0] - bins_net[0]
note_box(ax, f"weakest bin (6 to 8)\nmatched filter: {bins_mf[0]:.2f}\nnetwork: {bins_net[0]:.2f}\ngap: {gap:.2f}", x=0.97, y=0.55)
plt.show()

**What the result shows.** The filter recovers 1.00 of the signals in every bin. The network recovers 0.79, 0.92, 1.00, 1.00 and 1.00 in the five bins, centered at $\varrho$ = 7, 9, 11, 13 and 15.

| $\varrho$ | 7 | 9 | 11 | 13 | 15 |
|---|---|---|---|---|---|
| matched filter | 1.00 | 1.00 | 1.00 | 1.00 | 1.00 |
| network | 0.79 | 0.92 | 1.00 | 1.00 | 1.00 |

The gap closes above $\varrho \approx 10$ and is 0.21 in the weakest bin. This is the cost of a learned search in this regime: **it loses the marginal events**. They are the majority of the events of any catalog and a rate measurement depends on them.

The mock data challenge (Sch\"afer and others 2023) reports the sensitive volume and not the accuracy for this reason. The volume gives weight to the events that this plot shows to be lost.

## Part B. Your turn, at home

Part A compared the two methods in one setting. In Part B we change the setting, retrain the network and compare again. These are the functions we have defined for you. They are the functions of Part A and Part B defines no new one. Call them with keywords, as in Part A.

- `colored_noise(n_examples, psd=PSD)` returns segments of Gaussian noise with the one-sided PSD `psd`.
- `whiten_batch(x, psd=PSD)` returns the segments `x` whitened with `psd`, each with unit variance.
- `chirp_bank(masses, f_low=30.0, psd=PSD)` returns the bank of whitened chirps of unit norm, one for each mass.
- `sine_gaussian(centre, frequency, q)` returns a glitch on the time axis of a segment, with a maximum of 1.
- `make_set(n_examples, snr_lo=6.0, snr_hi=16.0, glitch_rate=0.0, bank=BANK)` returns the segments `x`, the labels `y` and the SNR of each chirp. The glitches are in the noise-only segments.
- `note_box(ax, text, x=0.97, y=0.95, ha='right')` writes a text in a box on the axes.
- `matched_filter_statistic(x, bank=BANK)` returns the largest $|\Phi|$ over the bank and over the arrival time, one value for each segment.
- `threshold_at(stat, y, false_alarm=0.01)` returns the threshold that the noise-only segments cross with probability `false_alarm`.
- `efficiency(stat, y, false_alarm=0.01)` returns the fraction of the signals above that threshold.
- `Net()` is the network. `train(x, y, epochs=10, seed=0, learning_rate=2e-3, batch_size=128)` returns the trained network.
- `score(net, x)` returns the score of each segment.
- `detection_curve(stat, y, false_alarms)` returns the efficiency at each false alarm probability of the list.
- `plot_efficiencies(labels, eff_mf, eff_net, ax=None)` draws pairs of bars, one pair for each case.
- `drifted_set(n_examples, tilt, snr_lo=6.0, snr_hi=16.0, bank=BANK)` returns segments `x` and labels `y` with a tilted noise spectrum, whitened with the old spectrum.
- `event_rank(stat, segment)` returns the place of a segment in the list ordered by the statistic.
- `efficiency_in_bins(stat, y, snr, edges, false_alarm=0.01, min_count=20)` returns the centers of the bins of the SNR and the efficiency in each.

**What we have from Part A.** `PSD`, `freqs`, `BANK`, `x_train`, `y_train`, `x_test`, `y_test`, `snr_test`, the filter statistic `mf_test`, the network `net` and its scores `cnn_test`, the efficiencies `eff_mf` and `eff_net` and the bins `edges`, `bins_mf` and `bins_net`. The random numbers are seeded in the setup cell. A different order of the cells gives different numbers.

### B1. The SNR of the training set

The network of A3 was trained on chirps with $\varrho$ between 6 and 16. Train a new network, with the same number of examples, on weaker chirps: $\varrho$ between 3 and 10. Print the efficiency at $F = 1$ % on the test set of A2, on all of it and in the weakest bin, from 6 to 8. Compare with the network of A3 and with the filter. What changes?

*Hint:* `make_set(6000, snr_lo=..., snr_hi=...)`, `train(x, y, seed=2)`, `score`, `efficiency` and `efficiency_in_bins(..., edges)`. Its second output is a list with one value for each bin.

In [ ]:
# your code here

> **Solution**
>
> Worked through in the session. Write your own version first — comparing
> the two is where the value is.


### B2. The false alarm probability

Compute the efficiency of both methods on the test set of A2 at $F = 0.1$, 0.03, 0.01 and 0.003 and print a table. Which method loses more when $F$ is smaller? What limits how small $F$ can be here?

*Hint:* `efficiency(stat, y, false_alarm=...)` with `mf_test` and `cnn_test`. Count how many noise-only segments the threshold rests on.

In [ ]:
# your code here

> **Solution**
>
> Worked through in the session. Write your own version first — comparing
> the two is where the value is.


### B3. The glitch rate

Write a prediction first: one sentence for each method on what the glitch rate does to its efficiency. Then build test sets with glitches in 10 %, 25 % and 50 % of the noise-only segments, score both methods (the network of A3, not retrained) and draw the efficiencies with `plot_efficiencies`. Is the prediction right?

*Hint:* `make_set(2000, glitch_rate=...)`, then `matched_filter_statistic` and `score(net, x)` on the segments. Then `efficiency` with the labels of that set.

In [ ]:
# your code here

> **Solution**
>
> Worked through in the session. Write your own version first — comparing
> the two is where the value is.


### B4. The size of the training set

Train the network on the first 500 and on the first 1500 segments of the training set and score the test set of A2. Plot the efficiency against the number of training examples, with the matched filter as a horizontal line. The network of A3 is the point at 6000. How many examples would the network need to match the filter? Is that practical?

*Hint:* `train(x_train[:500], y_train[:500])`, then `score` and `efficiency`.

In [ ]:
# your code here

> **Solution**
>
> Worked through in the session. Write your own version first — comparing
> the two is where the value is.


### B5. The bank

Run the matched filter on the test set of A2 with every second template of the bank, then every fourth and every eighth. The signals still come from all sixteen masses. Print the efficiency for 16, 8, 4 and 2 templates. What limits the loss? Does the network have a bank?

*Hint:* `BANK[::2]` is every second template. `matched_filter_statistic(x_test, bank=...)`.

In [ ]:
# your code here

> **Solution**
>
> Worked through in the session. Write your own version first — comparing
> the two is where the value is.


### B6. A network retrained on a changed spectrum (optional)

In A5 the network lost most efficiency at the tilt 0.6. Write a prediction, then train a new network on 6000 segments with that tilt. Score a test set with the same tilt with the new network, with the network of A3 and with the matched filter. Does the retraining recover the loss?

*Hint:* `drifted_set(6000, 0.6)` for the training set and `drifted_set(2000, 0.6)` for the test set. `train` takes the segments and the labels.

In [ ]:
# your code here

> **Solution**
>
> Worked through in the session. Write your own version first — comparing
> the two is where the value is.


## Software

`torch` for the network, `numpy` for the filter, `pycbc` for the waveforms.

The noise is **simulated** from the measured spectrum of the real Hanford data and the glitches are simulated too. Only A6 uses the real record, to show how little background a short stretch of data contains.

**References**

- Gabbard et al. (2018). *Matching Matched Filtering with Deep Networks for Gravitational-Wave Astronomy*.
- George and Huerta (2018). *Deep Learning for real-time gravitational wave detection and parameter estimation: Results with Advanced LIGO data*.
- Sch\"afer and others (2023). *First machine learning gravitational-wave search mock data challenge*.
- Cuoco et al. (2021). *Enhancing gravitational-wave science with machine learning*.